# Speech Emotion Recognition — data exploration

A quick look at RAVDESS: how the labels are stored, class balance, and what the
mel-spectrograms look like for different emotions.

## Setup

The emotion label is the 3rd field of each filename, e.g. `03-01-05-01-02-01-12.wav` → `05` → angry.

In [1]:
import os
import glob
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import IPython.display as ipd

DATA_DIR = os.path.join("..", "data", "raw")

EMOTIONS = {
    "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
    "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised",
}


def emotion_from_filename(path):
    return EMOTIONS[os.path.basename(path).split("-")[2]]


files = glob.glob(os.path.join(DATA_DIR, "Actor_*", "*.wav"))
len(files)

1440

## Class balance\n\nNeutral has half as many clips as the rest — worth remembering when reading the results.

In [ ]:
labels = [emotion_from_filename(f) for f in files]
unique, counts = np.unique(labels, return_counts=True)
dict(zip(unique, counts))

## One clip\n\nLoad a single file, look at its waveform, and listen to it.

In [ ]:
sample = files[100]
y, sr = librosa.load(sample, sr=22050)
print(os.path.basename(sample), "->", emotion_from_filename(sample))

plt.figure(figsize=(12, 3))
librosa.display.waveshow(y, sr=sr)
plt.title(emotion_from_filename(sample))
plt.tight_layout()
plt.show()

ipd.Audio(y, rate=sr)

## Mel-spectrogram

The waveform shows loudness over time but not pitch. A mel-spectrogram makes frequency
visible (y-axis), which is where a lot of the emotional signal lives.

In [ ]:
mel_db = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128), ref=np.max)

plt.figure(figsize=(12, 4))
librosa.display.specshow(mel_db, sr=sr, x_axis="time", y_axis="mel", cmap="magma")
plt.colorbar(format="%+2.0f dB")
plt.title(f"Mel-spectrogram — {emotion_from_filename(sample)}")
plt.tight_layout()
plt.show()

## Comparing emotions\n\nDifferent emotions have visibly different spectrograms — higher-energy emotions push more energy into the upper frequencies.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, emo in zip(axes.ravel(), ["calm", "happy", "angry", "sad"]):
    f = next(fp for fp in files if emotion_from_filename(fp) == emo)
    yy, _ = librosa.load(f, sr=sr)
    m = librosa.power_to_db(librosa.feature.melspectrogram(y=yy, sr=sr, n_mels=128), ref=np.max)
    librosa.display.specshow(m, sr=sr, x_axis="time", y_axis="mel", cmap="magma", ax=ax)
    ax.set_title(emo)

plt.tight_layout()
plt.show()